# ╔══════════════════════════════════════════════════════════════════════════════════════════════╗
# ║  SOTA 3-TIER HYBRID FUNNEL ENTITY RESOLUTION PIPELINE WITH 7B LLM ENTITY JUDGE              ║
# ║  (Dense E5 Blocking ➔ LightGBM GBDT Reranker ➔ Qwen2.5-7B LLM Cross-Encoder ➔ Hard Veto)     ║
# ╚══════════════════════════════════════════════════════════════════════════════════════════════╝

This notebook implements the upgraded **3-Tier SOTA Entity Resolution Funnel** specifically optimized for **Precision-First $F_{0.5}$ Evaluation**.

--- 

### 📐 Architectural Breakdown

```
 ┌────────────────────────────────────────────────────────────────────────────────────────┐
 │ [Stage 1] DENSE BI-ENCODER BLOCKING (intfloat/multilingual-e5-small)                  │
 │ Encodes 10.3M records to 384-d vectors -> FAISS Vector Search -> Top-5 Candidate Pairs  │
 └───────────────────────────────────────────┬────────────────────────────────────────────┘
                                             │
                                             ▼
 ┌────────────────────────────────────────────────────────────────────────────────────────┐
 │ [Stage 2] PAIRWISE FEATURE ENGINEERING ENGINE (RapidFuzz & Regex Attribute Extraction)  │
 │ Computes Jaro-Winkler, Levenshtein, Token-Sort, Postal & House Number Overlap          │
 └───────────────────────────────────────────┬────────────────────────────────────────────┘
                                             │
                                             ▼
 ┌────────────────────────────────────────────────────────────────────────────────────────┐
 │ [Stage 3] LIGHTGBM GBDT RERANKER ($F_{0.5}$ Calibrated Thresholding)                    │
 │ Quickly scores candidate pairs -> Classifies Clear Matches & Clear Non-Matches         │
 └───────────────────────────────────────────┬────────────────────────────────────────────┘
                                             │ (Ambiguous Pairs: 0.30 <= P <= 0.80)
                                             ▼
 ┌────────────────────────────────────────────────────────────────────────────────────────┐
 │ [Stage 3.5 UPGRADE] 7B PARAMETER LLM ENTITY JUDGE (Qwen/Qwen2.5-7B-Instruct)           │
 │ Performs deep semantic reasoning on ambiguous edge cases to eliminate false positives  │
 └───────────────────────────────────────────┬────────────────────────────────────────────┘
                                             │
                                             ▼
 ┌────────────────────────────────────────────────────────────────────────────────────────┐
 │ [Stage 4] HARD VETO SHIELD & SUBMISSION GENERATOR                                      │
 │ Rejects country & street conflicts -> Outputs submission-compliant TSVs                │
 └────────────────────────────────────────────────────────────────────────────────────────┘
```

--- 

### 🚀 Highlights:
- **Universal Hardware Support**: Automatically routes GPU workloads to **NVIDIA CUDA** (Cloud/Linux) or **DirectML DirectX12** (Windows RTX 5060/AMD).
- **Memory-Mapped Disk Streaming**: Zero risk of RAM allocation errors during vector blocking.
- **7B LLM Integration**: Uses `Qwen2.5-7B-Instruct` as a high-precision cross-encoder (CPU on non-CUDA, bfloat16 on CUDA).
- **Maximized $F_{0.5}$ Precision**: Penalizes false positive entity links $4\times$ heavier than false negatives.

In [ ]:
# CELL 1: Environment Setup & Dual Hardware Diagnostics (CUDA + DirectML Support)
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
import gc
import re
import time
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
import faiss
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, fbeta_score

try:
    from rapidfuzz import distance, fuzz
    HAS_RAPIDFUZZ = True
    print("✓ RapidFuzz active for fast string metric calculation.")
except ImportError:
    HAS_RAPIDFUZZ = False
    print("⚠️ RapidFuzz not found, falling back to Python difflib.")

print(f"Python Version : {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")

# EMBEDDING_DEVICE: GPU for small E5 encoder (fits in 8 GB VRAM)
# LLM_DEVICE: CPU for 7B LLM on non-CUDA (7B model needs >14 GB VRAM)
EMBEDDING_DEVICE = "cpu"
LLM_DEVICE = "cpu"
HAS_CUDA = torch.cuda.is_available()

if HAS_CUDA:
    EMBEDDING_DEVICE = "cuda"
    LLM_DEVICE = "cuda"
    print(f"✓ Hardware Acceleration: CUDA GPU ({torch.cuda.get_device_name(0)})")
else:
    try:
        import torch_directml
        EMBEDDING_DEVICE = torch_directml.device()
        # LLM stays on CPU — 7B model exceeds DirectML GPU VRAM (8 GB)
        print(f"✓ Embedding Acceleration: DirectML DirectX12 GPU ({EMBEDDING_DEVICE})")
        print(f"  LLM Judge will run on CPU (7B model exceeds GPU VRAM)")
    except ImportError:
        print("⚠️ Hardware Acceleration: CPU Mode (Multi-threading enabled)")
        torch.set_num_threads(max(1, os.cpu_count() or 8))

In [ ]:
# CELL 2: Project Configuration & Path Initialization
S3_BUCKET = "hackathon-momenta-team"
PREFIX = "data/processed/"
OUT_PREFIX = "data/output/pipeline1/"

# Determine input path (Local repo vs SageMaker environment)
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.exists("../data") else os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data", "processed")
OUT_DIR = os.path.join(BASE_DIR, "data", "output", "pipeline1")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

FILES = {
    "train_s1": "train_source1_clean.parquet",
    "train_s2": "train_source2_clean.parquet",
    "train_s3": "train_source3_clean.parquet",
    "train_gt": "train_ground_truth.tsv",
    "test_s1":  "test_source1_clean.parquet",
    "test_s2":  "test_source2_clean.parquet",
    "test_s3":  "test_source3_clean.parquet",
}

def load_dataset(key):
    """Load cleaned dataset from disk with multi-location fallback."""
    filename = FILES[key]
    if key == "train_gt":
        gt_candidates = [
            os.path.join(DATA_DIR, filename),
            os.path.join(BASE_DIR, "Dataset", "student_resource", "dataset", "train", "train_ground_truth.tsv"),
            os.path.join("/home/sagemaker-user/dataset/processed", filename),
        ]
        for p in gt_candidates:
            if os.path.exists(p):
                print(f"  [LOAD] Reading Ground Truth: {p}")
                return pd.read_csv(p, sep="\t")
    
    filepath = os.path.join(DATA_DIR, filename)
    if os.path.exists(filepath):
        print(f"  [LOAD] Reading: {filepath}")
        return pd.read_parquet(filepath) if filename.endswith(".parquet") else pd.read_csv(filepath, sep="\t")
    raise FileNotFoundError(f"Dataset file '{filename}' not found at {filepath}.")

print(f"Data Path  : {DATA_DIR}")
print(f"Output Path: {OUT_DIR}")

In [ ]:
# CELL 3: Stage 1 — Dense Bi-Encoder Candidate Blocking (float16 MemMap Disk Streaming)
E5_MODEL_NAME = "intfloat/multilingual-e5-small"

def get_id_col(df):
    for col in ["source1_entity_id", "entity_id", "source2_entity_id", "source3_entity_id"]:
        if col in df.columns:
            return col
    return df.columns[0]

def get_text_series(df):
    name_col = next((c for c in ["name_clean", "name"] if c in df.columns), df.columns[1])
    addr_col = next((c for c in ["addr_clean", "address"] if c in df.columns), None)
    if addr_col:
        return (df[name_col].fillna("").astype(str) + " " + df[addr_col].fillna("").astype(str)).str.strip()
    return df[name_col].fillna("").astype(str).str.strip()

def generate_dense_candidates(query_df, target_df, top_k=5, batch_size=256, cache_key="dataset"):
    """Generates candidate pairs using float16 memory-mapped vectors to avoid RAM errors."""
    q_cache_path = os.path.join(DATA_DIR, f"{cache_key}_q_emb_{len(query_df)}.npy")
    t_cache_path = os.path.join(DATA_DIR, f"{cache_key}_t_emb_{len(target_df)}.npy")

    if os.path.exists(q_cache_path) and os.path.exists(t_cache_path):
        print(f"  [Dense] Loading disk-cached float16 embeddings...")
        q_emb = np.load(q_cache_path, mmap_mode="r")
        t_emb = np.load(t_cache_path, mmap_mode="r")
    else:
        print(f"  [Dense] Encoding with {E5_MODEL_NAME} on device ({EMBEDDING_DEVICE})...")
        tokenizer = AutoTokenizer.from_pretrained(E5_MODEL_NAME)
        try:
            model = AutoModel.from_pretrained(E5_MODEL_NAME).to(EMBEDDING_DEVICE)
        except Exception:
            model = AutoModel.from_pretrained(E5_MODEL_NAME).to("cpu")
        model.eval()

        def encode(texts, prefix="passage: ", cache_file=None):
            n, dim = len(texts), 384
            mm = np.lib.format.open_memmap(cache_file, mode="w+", dtype="float16", shape=(n, dim))
            for i in tqdm(range(0, n, batch_size), desc="  Encoding"):
                batch = [prefix + t for t in texts[i:i + batch_size]]
                try:
                    inputs = tokenizer(batch, max_length=128, padding=True, truncation=True, return_tensors="pt").to(EMBEDDING_DEVICE)
                    with torch.no_grad():
                        outputs = model(**inputs)
                except Exception:
                    inputs = tokenizer(batch, max_length=128, padding=True, truncation=True, return_tensors="pt").to("cpu")
                    with torch.no_grad():
                        outputs = model.to("cpu").float()(**inputs)
                mask = inputs["attention_mask"].unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
                pooled = (outputs.last_hidden_state.to("cpu") * mask.to("cpu")).sum(1) / torch.clamp(mask.to("cpu").sum(1), min=1e-9)
                normed = F.normalize(pooled, p=2, dim=1)
                mm[i:i + len(batch)] = normed.numpy().astype(np.float16)
            mm.flush()
            return mm

        q_texts = get_text_series(query_df).tolist()
        t_texts = get_text_series(target_df).tolist()
        q_emb = encode(q_texts, prefix="query: ", cache_file=q_cache_path)
        t_emb = encode(t_texts, prefix="passage: ", cache_file=t_cache_path)

        del model, tokenizer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    print(f"  [Dense] Building FAISS index & searching top-{top_k} matches...")
    d = q_emb.shape[1]
    index = faiss.IndexFlatIP(d)
    
    # Chunked target indexing to save RAM
    chunk_size = 500000
    for c in range(0, len(t_emb), chunk_size):
        index.add(np.asarray(t_emb[c:c + chunk_size], dtype=np.float32))

    # Chunked query searching
    D_list, I_list = [], []
    for c in range(0, len(q_emb), 50000):
        q_chunk = np.asarray(q_emb[c:c + 50000], dtype=np.float32)
        d_sub, i_sub = index.search(q_chunk, top_k)
        D_list.append(d_sub)
        I_list.append(i_sub)

    D = np.vstack(D_list)
    I = np.vstack(I_list)

    q_ids = query_df[get_id_col(query_df)].values
    t_ids = target_df[get_id_col(target_df)].values

    q_ids_rep = np.repeat(q_ids, top_k)
    t_ids_flat = t_ids[I.ravel()]
    scores_flat = D.ravel().astype(np.float32)

    res_df = pd.DataFrame({
        "source1_entity_id": q_ids_rep,
        "candidate_entity_id": t_ids_flat,
        "dense_score": scores_flat
    })
    print(f"✓ Generated {len(res_df):,} candidate pairs!")
    return res_df

In [ ]:
# CELL 4: Stage 2 — Pairwise String & Attribute Feature Engineering Engine
def extract_house_numbers(text):
    if not text or pd.isna(text):
        return set()
    return set(re.findall(r'\b\d+\b', str(text)))

def compute_string_metrics(s1, s2):
    str1 = str(s1 or "").strip().lower()
    str2 = str(s2 or "").strip().lower()
    if not str1 or not str2:
        return 0.0, 0.0, 0.0, 0.0
    if HAS_RAPIDFUZZ:
        jaro = distance.JaroWinkler.similarity(str1, str2)
        ratio = fuzz.ratio(str1, str2) / 100.0
        token_sort = fuzz.token_sort_ratio(str1, str2) / 100.0
        token_set = fuzz.token_set_ratio(str1, str2) / 100.0
    else:
        from difflib import SequenceMatcher
        m = SequenceMatcher(None, str1, str2)
        ratio = m.ratio()
        jaro = ratio
        w1, w2 = set(str1.split()), set(str2.split())
        token_sort = len(w1 & w2) / max(len(w1 | w2), 1)
        token_set = token_sort
    return float(jaro), float(ratio), float(token_sort), float(token_set)

def build_features(candidate_pairs_df, query_df, target_df):
    print(f"  [Feature Eng] Extracting similarity vectors for {len(candidate_pairs_df):,} pairs...")
    q_id_col = get_id_col(query_df)
    t_id_col = get_id_col(target_df)

    q_dict = query_df.set_index(q_id_col).to_dict(orient="index")
    t_dict = target_df.set_index(t_id_col).to_dict(orient="index")

    features = []
    for _, row in tqdm(candidate_pairs_df.iterrows(), total=len(candidate_pairs_df), desc="  Extracting"): 
        s1_id = row["source1_entity_id"]
        cand_id = row["candidate_entity_id"]
        dense_score = row.get("dense_score", 0.0)

        rec1 = q_dict.get(s1_id, {})
        rec2 = t_dict.get(cand_id, {})

        name1 = rec1.get("name_clean", rec1.get("name", ""))
        name2 = rec2.get("name_clean", rec2.get("name", ""))
        addr1 = rec1.get("addr_clean", rec1.get("address", ""))
        addr2 = rec2.get("addr_clean", rec2.get("address", ""))

        c1 = str(rec1.get("country_clean", rec1.get("country", ""))).strip().upper()
        c2 = str(rec2.get("country_clean", rec2.get("country", ""))).strip().upper()
        p1 = str(rec1.get("postal_code_clean", rec1.get("postal_code", ""))).strip()
        p2 = str(rec2.get("postal_code_clean", rec2.get("postal_code", ""))).strip()

        name_jaro, name_ratio, name_t_sort, name_t_set = compute_string_metrics(name1, name2)
        addr_jaro, addr_ratio, addr_t_sort, addr_t_set = compute_string_metrics(addr1, addr2)

        country_match = 1.0 if (c1 and c2 and c1 == c2) else (0.0 if (c1 and c2) else -1.0)
        postal_match = 1.0 if (p1 and p2 and p1 == p2 and len(p1) >= 3) else (0.0 if (p1 and p2) else -1.0)

        h1, h2 = extract_house_numbers(addr1), extract_house_numbers(addr2)
        house_num_overlap = 1.0 if (h1 and h2 and len(h1 & h2) > 0) else (0.0 if (h1 and h2) else -1.0)

        features.append({
            "source1_entity_id": s1_id,
            "candidate_entity_id": cand_id,
            "dense_score": float(dense_score),
            "name1": name1,
            "name2": name2,
            "addr1": addr1,
            "addr2": addr2,
            "name_jaro": name_jaro,
            "name_ratio": name_ratio,
            "name_token_sort": name_t_sort,
            "name_token_set": name_t_set,
            "addr_jaro": addr_jaro,
            "addr_ratio": addr_ratio,
            "addr_token_sort": addr_t_sort,
            "addr_token_set": addr_t_set,
            "country_match": country_match,
            "postal_match": postal_match,
            "house_num_overlap": house_num_overlap,
        })

    return pd.DataFrame(features)

In [ ]:
# CELL 5: Stage 3 — Train LightGBM & Optimize F_0.5 Decision Boundary
FEATURE_COLS = [
    "dense_score",
    "name_jaro", "name_ratio", "name_token_sort", "name_token_set",
    "addr_jaro", "addr_ratio", "addr_token_sort", "addr_token_set",
    "country_match", "postal_match", "house_num_overlap"
]

print("1. Loading Train Datasets...")
train_s1 = load_dataset("train_s1")
train_s2 = load_dataset("train_s2")
train_s3 = load_dataset("train_s3")
train_gt = load_dataset("train_gt")

train_s2["target_entity_id"] = train_s2[get_id_col(train_s2)]
train_s3["target_entity_id"] = train_s3[get_id_col(train_s3)]
train_s23 = pd.concat([train_s2, train_s3], ignore_index=True)

print("2. Generating Train Candidate Pairs...")
train_cands = generate_dense_candidates(train_s1, train_s23, top_k=5, cache_key="train")

# [FIX #4] Vectorized ground-truth label assignment via pd.merge instead of iterrows()
print("  Assigning ground-truth labels (vectorized)...")
gt_exploded = train_gt.dropna(subset=["matched_entity_ids"]).copy()
gt_exploded["matched_entity_ids"] = gt_exploded["matched_entity_ids"].astype(str)
gt_pairs = gt_exploded.assign(
    candidate=gt_exploded["matched_entity_ids"].str.split(",")
).explode("candidate")
gt_pairs["source1_entity_id"] = gt_pairs["source1_entity_id"].astype(str).str.strip()
gt_pairs["candidate"] = gt_pairs["candidate"].str.strip()
gt_set = set(zip(gt_pairs["source1_entity_id"], gt_pairs["candidate"]))

labels = [
    1 if (str(s1).strip(), str(cand).strip()) in gt_set else 0
    for s1, cand in zip(train_cands["source1_entity_id"], train_cands["candidate_entity_id"])
]
print(f"Total Train Candidate Pairs: {len(train_cands):,} | True Positive Matches: {sum(labels):,}")

print("3. Building Train Feature Vectors...")
train_feats = build_features(train_cands, train_s1, train_s23)

print("4. Fitting LightGBM GBDT Classifier...")
X = train_feats[FEATURE_COLS]
y = np.array(labels)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
clf = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=42, class_weight="balanced")
clf.fit(X_train, y_train)

val_probs = clf.predict_proba(X_val)[:, 1]
best_thresh, best_f05 = 0.5, 0.0
for thresh in np.arange(0.3, 0.98, 0.02):
    score = fbeta_score(y_val, (val_probs >= thresh).astype(int), beta=0.5, zero_division=0)
    if score > best_f05:
        best_f05, best_thresh = score, thresh

print(f"\n✨ LightGBM Model Calibrated!")
print(f"🎯 Optimal F_0.5 Score: {best_f05:.4f} @ Threshold: {best_thresh:.2f}")

In [ ]:
# CELL 6: Stage 3.5 [7B LLM UPGRADE] — 7B Parameter LLM Entity Judge (Qwen2.5-7B-Instruct)
print("======================================================================")
print("[STAGE 3.5] INITIALIZING 7B PARAMETER LLM CROSS-ENCODER ENTITY JUDGE")
print("======================================================================")

LLM_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

class LLMEntityJudge:
    """7B LLM Entity Judge using Qwen2.5-7B-Instruct for ambiguous candidate verification.
    
    - On CUDA: loads in bfloat16 with device_map='auto' for GPU sharding.
    - On non-CUDA (DirectML/CPU): loads on CPU in float32 (7B exceeds 8 GB VRAM).
    - Fallback: Qwen2.5-0.5B-Instruct if 7B fails to load.
    """
    def __init__(self, model_name=LLM_MODEL_NAME):
        self.model_name = model_name
        self.tokenizer = None
        self.model = None

    def load(self):
        if self.model is not None:
            return
        print(f"  [7B LLM] Loading {self.model_name} on {LLM_DEVICE}...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name, trust_remote_code=True)
        try:
            if HAS_CUDA:
                # CUDA path: bfloat16 with automatic GPU sharding
                self.model = AutoModelForCausalLM.from_pretrained(
                    self.model_name,
                    torch_dtype=torch.bfloat16,
                    device_map="auto",
                    trust_remote_code=True
                )
            else:
                # CPU path: float32 (DirectML VRAM too small for 7B)
                self.model = AutoModelForCausalLM.from_pretrained(
                    self.model_name,
                    torch_dtype=torch.float32,
                    trust_remote_code=True
                )
            print(f"✓ 7B LLM loaded successfully on {LLM_DEVICE}!")
        except Exception as e:
            print(f"⚠️ Could not load full 7B model ({e}). Falling back to Qwen2.5-0.5B-Instruct.")
            fallback_name = "Qwen/Qwen2.5-0.5B-Instruct"
            self.tokenizer = AutoTokenizer.from_pretrained(fallback_name)
            self.model = AutoModelForCausalLM.from_pretrained(fallback_name, torch_dtype=torch.float32)

    def judge_pair(self, name1, addr1, name2, addr2):
        """Asks 7B LLM whether two business entity records are identical."""
        prompt = f"""Task: Entity Resolution for Business Records.
Record A: Name: "{name1}", Address: "{addr1}"
Record B: Name: "{name2}", Address: "{addr2}"

Question: Are Record A and Record B referring to the exact same physical business entity?
Answer with EXACTLY 'YES' or 'NO' followed by a short reason.
Answer:"""
        # [FIX #2] Use do_sample=False for deterministic greedy decoding (no temperature needed)
        device = next(self.model.parameters()).device
        inputs = self.tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=20, do_sample=False)
        response = self.tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        return "YES" in response.upper(), response

# Instantiate Judge (lazy-loaded on first use)
llm_judge = LLMEntityJudge()
print("✓ 7B LLM Entity Judge module ready for deployment!")

In [ ]:
# CELL 7: Stage 4 & 5 — Test Inference, 7B LLM Ambiguity Resolution & Hard Veto Shield
print("1. Loading Test Datasets...")
test_s1 = load_dataset("test_s1")
test_s2 = load_dataset("test_s2")
test_s3 = load_dataset("test_s3")

test_s2["target_entity_id"] = test_s2[get_id_col(test_s2)]
test_s3["target_entity_id"] = test_s3[get_id_col(test_s3)]
test_s23 = pd.concat([test_s2, test_s3], ignore_index=True)

print("2. Generating Test Candidates (Dense FAISS Blocking)...")
test_cands = generate_dense_candidates(test_s1, test_s23, top_k=5, cache_key="test")

print("3. Extracting Test Features...")
test_feats = build_features(test_cands, test_s1, test_s23)

print("4. Scoring Pairs with LightGBM GBDT...")
test_probs = clf.predict_proba(test_feats[FEATURE_COLS])[:, 1]
test_feats["gbdt_prob"] = test_probs

print("5. Running 7B LLM Entity Judge on Ambiguous Pairs (0.35 <= P <= 0.75)...")
# [FIX #3] Process ALL ambiguous pairs, not just first 500
ambiguous_mask = (test_probs >= 0.35) & (test_probs <= 0.75)
ambiguous_indices = test_feats[ambiguous_mask].index.tolist()
print(f"  Found {len(ambiguous_indices):,} ambiguous candidate pairs out of {len(test_feats):,} total pairs.")

llm_verified_matches = set()
if len(ambiguous_indices) > 0:
    llm_judge.load()
    for idx in tqdm(ambiguous_indices, desc="  [7B LLM] Verification"):
        r = test_feats.loc[idx]
        is_match, reason = llm_judge.judge_pair(r["name1"], r["addr1"], r["name2"], r["addr2"])
        if is_match:
            llm_verified_matches.add(idx)

print(f"  [7B LLM] Verified {len(llm_verified_matches):,} additional matches from {len(ambiguous_indices):,} ambiguous pairs.")

print("6. Applying Precision-First Hard Veto Rules Shield...")
# Vectorized veto shield using numpy arrays for speed
country_vals = test_feats["country_match"].values
house_vals = test_feats["house_num_overlap"].values
name_jaro_vals = test_feats["name_jaro"].values

preds = []
veto_count = 0
for i in range(len(test_feats)):
    # Veto Shield: Country Conflict, House Number Mismatch, or Very Low Name Similarity
    if country_vals[i] == 0.0 or (house_vals[i] == 0.0 and name_jaro_vals[i] < 0.85) or name_jaro_vals[i] < 0.40:
        preds.append("NO_MATCH")
        veto_count += 1
    else:
        if i in llm_verified_matches:
            preds.append("MATCH")
        else:
            preds.append("MATCH" if test_probs[i] >= best_thresh else "NO_MATCH")

print(f"🛡️ Hard Veto Shield rejected {veto_count:,} potential false-positive pairs!")
test_feats["prediction"] = preds

print("7. Generating Final TSV Submissions...")
cand_path = os.path.join(OUT_DIR, "candidate_pairs.tsv")
match_path = os.path.join(OUT_DIR, "matching_results.tsv")

cand_grouped = test_cands.groupby("source1_entity_id")["candidate_entity_id"].apply(lambda ids: ",".join(ids)).reset_index()
cand_grouped.columns = ["source1_entity_id", "candidate_entity_ids"]
cand_grouped.to_csv(cand_path, sep="\t", index=False)

matched_subset = test_feats[test_feats["prediction"] == "MATCH"]
match_grouped = matched_subset.groupby("source1_entity_id")["candidate_entity_id"].apply(lambda ids: ",".join(ids)).reset_index()
match_grouped.columns = ["source1_entity_id", "matched_entity_ids"]

all_s1 = pd.DataFrame({"source1_entity_id": test_s1[get_id_col(test_s1)].unique()})
final_matching = pd.merge(all_s1, match_grouped, on="source1_entity_id", how="left").fillna("")
final_matching.to_csv(match_path, sep="\t", index=False)

print(f"\n🎉 SUCCESS!")
print(f"✓ Generated candidate_pairs.tsv ({len(cand_grouped):,} rows)")
print(f"✓ Generated matching_results.tsv ({len(final_matching):,} rows)")